# Supplementary Figure S4: Full Stratified Robustness

- Figure / panels: `FigS4a-d` stratified robustness, `FigS4e` split-wise global structure recovery
- Inputs: metrics and payloads for the selected datasets
- Outputs: `artifacts/paper_figures/supp/FigS4_Robustness/*`
- Run order: optional; run after Fig3 if the full robustness supplement is needed
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Supplementary


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style
import scripts.trishift.analysis.split_structure_recovery as split_structure_recovery
import scripts.trishift.analysis.stratified_benchmark as stratified_benchmark

apply_gears_paper_style(font_scale=1.0)
importlib.reload(stratified_benchmark)
importlib.reload(split_structure_recovery)



In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']  # edit here
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]

STRUCTURE_PANEL_ENABLED = True  # edit here
STRUCTURE_PANEL_DATASET = 'norman'  # edit here
STRUCTURE_PANEL_MODELS = ['trishift_nearest']  # edit here
STRUCTURE_PANEL_MAX_CELLS_PER_CONDITION = 28
STRUCTURE_PANEL_CLUSTER_K = 4

OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS4_Robustness'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_frames = []
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Selected models:', MODELS)
for spec in DATASET_SPECS:
    try:
        result = stratified_benchmark.run_stratified_benchmark(dataset=spec['dataset_key'], models=MODELS, split_ids=SPLIT_IDS, out_root=OUT_ROOT / spec['dataset_key'])
    except Exception:
        continue
    df = result['summary_df'].copy(); df['dataset_label'] = spec['dataset_label']; summary_frames.append(df)
summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary_df.to_csv(OUT_ROOT / 'figs4_summary_all.csv', index=False, encoding='utf-8-sig')
display(summary_df.head())

if STRUCTURE_PANEL_ENABLED:
    structure_spec = next((spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] == STRUCTURE_PANEL_DATASET), None)
    if structure_spec is None:
        raise ValueError(f'Unknown STRUCTURE_PANEL_DATASET: {STRUCTURE_PANEL_DATASET}')
    if STRUCTURE_PANEL_DATASET not in SELECTED_DATASET_KEYS:
        print(f'Skip structure panel because {STRUCTURE_PANEL_DATASET!r} is not in SELECTED_DATASET_KEYS')
    else:
        structure_result = split_structure_recovery.run_split_structure_recovery(
            dataset=STRUCTURE_PANEL_DATASET,
            dataset_label=structure_spec['dataset_label'],
            models=STRUCTURE_PANEL_MODELS,
            split_ids=SPLIT_IDS,
            out_root=OUT_ROOT,
            max_cells_per_condition=STRUCTURE_PANEL_MAX_CELLS_PER_CONDITION,
            cluster_k=STRUCTURE_PANEL_CLUSTER_K,
        )
        print(structure_result.figure_path)
        if structure_result.figure_path.exists():
            display(Image(filename=str(structure_result.figure_path), width=1200))
        display(structure_result.summary_df.head())

print(OUT_ROOT)
